In [1]:
# Cell 1 - Non-destructive multi-source profiling and cross-source schema/parity audit
from pathlib import Path
import re
import numpy as np
import pandas as pd
import openpyxl

SEED = 42
np.random.seed(SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# 1. Raw file inventory across all 5 source folders
inv_rows = []
for folder in sorted([p for p in RAW_DIR.iterdir() if p.is_dir()]):
    for fp in sorted(folder.iterdir()):
        if fp.is_file():
            n_rows, n_cols = None, None
            if fp.suffix.lower() == ".csv":
                df_head = pd.read_csv(fp)
                n_rows, n_cols = df_head.shape
            inv_rows.append({
                "folder": folder.name,
                "file": fp.name,
                "ext": fp.suffix.lower(),
                "kb": round(fp.stat().st_size / 1024.0, 1),
                "rows": n_rows,
                "cols": n_cols,
            })

df_inv = pd.DataFrame(inv_rows)
print(f"step=raw_inventory | files={len(df_inv)} | total_kb={df_inv['kb'].sum():.1f} | csv_rows={int(df_inv['rows'].dropna().sum())}")
print(df_inv.groupby("folder")[["file", "kb", "rows"]].agg({"file": "count", "kb": "sum", "rows": "sum"}).to_string())

# 2. Audit OJK SJKPublic provincial tables and verify TWP90 denominator anomaly (144 vs 1453)
sjk_dir = RAW_DIR / "ojk_sjkpublic"
df_konv_twp = pd.read_csv(sjk_dir / "144_konv_twp90_dati1_LPBBTI_TWP90_DATI1.csv")
df_syar_twp = pd.read_csv(sjk_dir / "312_syar_twp90_dati1_LPBBTI_Syariah_TWP90_DATI1.csv")
df_konv_qual = pd.read_csv(sjk_dir / "1453_konv_borrower_kualitas_dati1_LPBBTI_PenerimaPinjamanKualitasPerDati.csv")
df_syar_qual = pd.read_csv(sjk_dir / "1454_syar_borrower_kualitas_dati1_LPBBTI_Syariah_PenerimaPinjamanKualitasPerDati.csv")
df_konv_bor = pd.read_csv(sjk_dir / "411_konv_borrower_dati1_LPBBTI_PenerimaPinjamanDATI1.csv")
df_konv_rek = pd.read_csv(sjk_dir / "417_konv_rekening_dati1_LPBBTI_InfoKontrakUsahadanRekeningDATI1Penerima.csv")

print("\nstep=audit_sjkpublic_nulls_and_entities")
for name, d in [
    ("konv_twp90", df_konv_twp), ("syar_twp90", df_syar_twp),
    ("konv_qual", df_konv_qual), ("konv_bor", df_konv_bor), ("konv_rek", df_konv_rek)
]:
    all_null_cols = [c for c in d.columns if d[c].isna().all()]
    part_null_cols = {c: int(d[c].isna().sum()) for c in d.columns if 0 < d[c].isna().sum() < len(d)}
    print(f"  table={name} | rows={len(d)} | months={d['Bulan'].min()}..{d['Bulan'].max()} | all_null={all_null_cols} | part_null={part_null_cols}")

print(f"  unique_NamaDATI1_count={df_konv_twp['NamaDATI1'].nunique()} (includes 001. Jawa, 008. Luar Jawa aggregates)")
print("  unique_KualitasPembiayaan=", sorted(df_konv_qual["KualitasPembiayaan"].unique()))

# 3. Audit OJK Historical XLSX Sheet 9 (note: aggregates live in col A, provinces in col B)
xlsx_dir = RAW_DIR / "ojk_xlsx"
wb_2306 = openpyxl.load_workbook(xlsx_dir / "ojk_lpbbti_2023_06.xlsx", read_only=True, data_only=True)
wb_2512 = openpyxl.load_workbook(xlsx_dir / "ojk_lpbbti_2025_12.xlsx", read_only=True, data_only=True)

def inspect_sheet9_locs(wb):
    s9 = [s for s in wb.sheetnames if s.strip() == "9"][0]
    ws = wb[s9]
    rows_all = list(ws.iter_rows(min_row=1, max_row=65, values_only=True))
    hdr_dates = [c for c in rows_all[1] if c is not None]
    locs = []
    sample_vals = {}
    for row in rows_all[3:]:
        c0 = str(row[0]).strip() if row[0] is not None else ""
        c1 = str(row[1]).strip() if len(row) > 1 and row[1] is not None else ""
        label = c1 if c1 else c0
        if not label or "jumlah" in label.lower() or "total" in label.lower() or "catatan" in label.lower() or "note" in label.lower():
            if label:
                locs.append(("AGG/NOTE", label))
            continue
        is_agg = bool(c0 and not c1)
        locs.append(("AGG" if is_agg else "PROV", label))
        if "Jawa Timur" in label or "DKI Jakarta" in label:
            sample_vals[label] = (row[2], row[3], row[4])
    return hdr_dates, locs, sample_vals

d23, locs23, samp23 = inspect_sheet9_locs(wb_2306)
d25, locs25, samp25 = inspect_sheet9_locs(wb_2512)
wb_2306.close()
wb_2512.close()

prov23 = [lbl for kind, lbl in locs23 if kind == "PROV"]
prov25 = [lbl for kind, lbl in locs25 if kind == "PROV"]
print("\nstep=audit_ojk_xlsx_sheet9_and_parity")
print(f"  2023_06_dates={d23[:4]}..{d23[-1:]} | n_date_cols={len(d23)-1} | n_prov_rows={len(prov23)}")
print(f"  2025_12_dates={d25[:3]}..{d25[-1:]} | n_date_cols={len(d25)-1} | n_prov_rows={len(prov25)}")
print(f"  2025_12_typo_samples={[x for x in prov25 if any(k in x for k in ['Nangroe', 'Kepualauan', 'Papua'])]}")

# Prove denominator anomaly in SJKPublic table 144 vs true provincial TWP90 from 1453+1454 & XLSX Sheet 9
sub144 = df_konv_twp[(df_konv_twp["Bulan"] == "2024-12") & (~df_konv_twp["NamaDATI1"].isin(["001. Jawa", "008. Luar Jawa"]))]
piv_k = df_konv_qual[df_konv_qual["Bulan"] == "2024-12"].pivot_table(
    index="DATI1", columns="KualitasPembiayaan", values="OutstandingPerseoranganKualitasPerDati", aggfunc="sum"
).fillna(0)
piv_s = df_syar_qual[df_syar_qual["Bulan"] == "2024-12"].pivot_table(
    index="DATI1", columns="KualitasPembiayaan", values="OutstandingPerseoranganKualitasPerDati", aggfunc="sum"
).fillna(0)

jt_k_tot = piv_k.loc["007. Jawa Timur"].sum()
jt_k_macet = piv_k.loc["007. Jawa Timur", "005. Pendanaan Macet"]
jt_s_tot = piv_s.loc["007. Jawa Timur"].sum()
jt_s_macet = piv_s.loc["007. Jawa Timur", "005. Pendanaan Macet"]
nat_k_tot = piv_k.sum().sum()

jt_true_konv_twp90 = jt_k_macet / jt_k_tot
jt_true_comb_twp90 = (jt_k_macet + jt_s_macet) / (jt_k_tot + jt_s_tot)
jt_nat_share_twp90 = (jt_k_macet / nat_k_tot) * 100.0
jt_144_val = float(sub144.loc[sub144["NamaDATI1"] == "007. Jawa Timur", "TWP90"].iloc[0])
jt_xlsx_val = samp25.get("6. Jawa Timur")

print("  parity_JawaTimur_2024_12:")
print(f"    xlsx_sheet9(rek, out_bn, twp90_ratio) = {jt_xlsx_val}")
print(f"    sjk_1453+1454_combined(out_bn={((jt_k_tot+jt_s_tot)/1e9):.6f}, twp90_ratio={jt_true_comb_twp90:.8f})")
print(f"    sjk_1453_konv_only_twp90_ratio        = {jt_true_konv_twp90:.8f}")
print(f"    sjk_144_reported_TWP90={jt_144_val:.4f} vs (macet_prov / nat_outstanding * 100)={jt_nat_share_twp90:.4f}")
print(f"    sjk_144_sum_across_provinces={sub144['TWP90'].sum():.4f} (equals national Konv TWP90 pct)")

# 4. Audit BPS raw CSVs
bps_dir = RAW_DIR / "bps"
print("\nstep=audit_bps_tables")
for fn in sorted(bps_dir.glob("*.csv")):
    d = pd.read_csv(fn)
    prov_col = "provinsi_raw" if "provinsi_raw" in d.columns else "province_name"
    val_col = "nilai_raw" if "nilai_raw" in d.columns else "amount_idr"
    year_col = "tahun" if "tahun" in d.columns else "year"
    vals_str = [str(v) for v in d[val_col].dropna().tolist()]
    dirty_vals = [v for v in vals_str if re.search(r"[^\d.\-]", v)]
    print(f"  file={fn.name} | rows={len(d)} | prov_uniq={d[prov_col].nunique()} | years={sorted(d[year_col].dropna().unique().tolist())[:6]} | dirty_num_vals={len(dirty_vals)}")
    if dirty_vals:
        print(f"    sample_dirty_vals={list(dict.fromkeys(dirty_vals))[:6]}")

# 5. Audit OJK Directory, Play Store Matches, Reviews, and Satgas PASTI
df_dir = pd.read_csv(RAW_DIR / "ojk_directory" / "ojk_direktori_lpbbti_mei2026_raw.csv")
df_apps = pd.read_csv(RAW_DIR / "playstore" / "playstore_pindar_apps_raw.csv")
df_rev = pd.read_csv(RAW_DIR / "playstore" / "playstore_pindar_reviews_raw.csv")
df_ill = pd.read_csv(RAW_DIR / "playstore" / "satgas_pasti_pinjol_ilegal_okt2026_raw.csv")

def brand_title_mismatch(row):
    b = re.sub(r"[^a-z0-9]", "", str(row["nama_sistem_elektronik"]).lower())
    t = re.sub(r"[^a-z0-9]", "", str(row["title"]).lower())
    return b not in t and t not in b

suspicious_apps = df_apps[df_apps.apply(brand_title_mismatch, axis=1)][
    ["no_ojk", "nama_sistem_elektronik", "nama_perusahaan", "appId", "title", "developer", "match_rule"]
]
print("\nstep=audit_directory_playstore_satgas")
print(f"  ojk_dir_rows={len(df_dir)} | os_anomalies={df_dir['sistem_operasi'].value_counts().to_dict()}")
print(f"  playstore_apps={len(df_apps)} | duplicate_appIds={df_apps['appId'].duplicated().sum()} | brand_title_mismatches={len(suspicious_apps)}")
print(suspicious_apps.to_string(index=False))
print(f"  reviews_rows={len(df_rev)} | duplicate_reviewIds={df_rev['reviewId'].duplicated().sum()} | null_content={df_rev['content'].isna().sum()} | null_reply={df_rev['replyContent'].isna().sum()}")
print(f"  illegal_pinjol_rows={len(df_ill)} | nulls={df_ill.isna().sum().to_dict()}")


step=raw_inventory | files=55 | total_kb=12452.3 | csv_rows=29173
               file      kb     rows
folder                              
bps              11   947.1   6314.0
ojk_directory     2   305.9     94.0
ojk_sjkpublic    19  1904.7  18223.0
ojk_xlsx         18  7201.2      0.0
playstore         5  2093.4   4542.0

step=audit_sjkpublic_nulls_and_entities
  table=konv_twp90 | rows=880 | months=2024-07..2026-04 | all_null=['TWP90_real'] | part_null={'Semester': 760, 'Quartal': 600}


  table=syar_twp90 | rows=880 | months=2024-07..2026-04 | all_null=['TWP90_real'] | part_null={'Semester': 760, 'Quartal': 600}
  table=konv_qual | rows=4180 | months=2024-07..2026-04 | all_null=['OutstandingPerseoranganKualitasPerDati_real'] | part_null={'Semester': 3610, 'Quartal': 2850}
  table=konv_bor | rows=902 | months=2024-07..2026-04 | all_null=['JumlahDanaBrdsrknDATI1_real', 'AkumulasiDanaBrdsrknDATI1_real'] | part_null={'Semester': 779, 'Quartal': 615}
  table=konv_rek | rows=1066 | months=2024-07..2026-08 | all_null=['AkumulasiJumlahRekeningPenerimaPinjaman', 'JumlahRekeningPenerimaPinjaman_real', 'AkumulasiJumlahRekeningPenerimaPinjaman_real'] | part_null={'Semester': 902, 'Quartal': 738}
  unique_NamaDATI1_count=40 (includes 001. Jawa, 008. Luar Jawa aggregates)
  unique_KualitasPembiayaan= ['001. Pendanaan Lancar', '002. Pendanaan Dalam Perhatian Khusus', '003. Pendanaan Kurang Lancar', '004. Pendanaan Diragukan', '005. Pendanaan Macet']



step=audit_ojk_xlsx_sheet9_and_parity
  2023_06_dates=['Lokasi / Locations', '*Jan-22', '*Feb-22', datetime.datetime(2022, 3, 1, 0, 0)]..[datetime.datetime(2023, 6, 1, 0, 0)] | n_date_cols=18 | n_prov_rows=35
  2025_12_dates=['Lokasi / Locations', datetime.datetime(2024, 12, 1, 0, 0), datetime.datetime(2025, 1, 1, 0, 0)]..[datetime.datetime(2025, 12, 1, 0, 0)] | n_date_cols=13 | n_prov_rows=38
  2025_12_typo_samples=['1. Nangroe Aceh Darussalam', '6. Kepualauan Bangka Belitung', '27. Papua Barat', '28. Papua', '29. Papua Tengah', '30. Papua Pegunungan', '31. Papua Selatan', '32. Papua Barat Daya']
  parity_JawaTimur_2024_12:
    xlsx_sheet9(rek, out_bn, twp90_ratio) = (2644816, 9573.792940496, 0.028941072294242164)
    sjk_1453+1454_combined(out_bn=9573.792940, twp90_ratio=0.02894107)
    sjk_1453_konv_only_twp90_ratio        = 0.02904059
    sjk_144_reported_TWP90=0.3643 vs (macet_prov / nat_outstanding * 100)=0.3644
    sjk_144_sum_across_provinces=2.6067 (equals national Konv TWP90


step=audit_directory_playstore_satgas
  ojk_dir_rows=94 | os_anomalies={'Android': 37, 'Android & iOS': 32, '-': 24, 'Andorid': 1}
  playstore_apps=72 | duplicate_appIds=3 | brand_title_mismatches=7
 no_ojk nama_sistem_elektronik                 nama_perusahaan                     appId                         title                      developer match_rule
     12                 Ammana        PT Ammana Fintek Syariah          com.alami_funder         ALAMI Funding Syariah         PT ALAMI Fintek Sharia   comp_dev
     14                KoinP2P      PT Lunaria Annua Teknologi         com.koinworks.app KoinWorks Pendanaan Super App    PT. LUNARIA ANNUA TEKNOLOGI   comp_dev
     27          OVO Finansial      PT Indonusa Bara Sejahtera             com.asf.singa Singa Fintech-Pinjaman Daring  PT Abadi Sejahtera Finansindo   comp_dev
     30              AwanTunai PT Simplefi Teknologi Indonesia         com.awantunai.app AwanToko: Warung Makin Untung       PT. Awan Tunai Indonesia  pkg_b

In [2]:
# Cell 2 - Harmonize OJK historical XLSX (2022-2025) and SJKPublic (2024-2026) provincial panels
import datetime

CANONICAL_38_PROVINCES = {
    "aceh": ("11", "Aceh", "Sumatera"),
    "nangroe aceh darussalam": ("11", "Aceh", "Sumatera"),
    "nanggroe aceh darussalam": ("11", "Aceh", "Sumatera"),
    "sumatera utara": ("12", "Sumatera Utara", "Sumatera"),
    "sumatera barat": ("13", "Sumatera Barat", "Sumatera"),
    "riau": ("14", "Riau", "Sumatera"),
    "jambi": ("15", "Jambi", "Sumatera"),
    "sumatera selatan": ("16", "Sumatera Selatan", "Sumatera"),
    "bengkulu": ("17", "Bengkulu", "Sumatera"),
    "lampung": ("18", "Lampung", "Sumatera"),
    "kepulauan bangka belitung": ("19", "Kepulauan Bangka Belitung", "Sumatera"),
    "kepualauan bangka belitung": ("19", "Kepulauan Bangka Belitung", "Sumatera"),
    "kep. bangka belitung": ("19", "Kepulauan Bangka Belitung", "Sumatera"),
    "kepulauan riau": ("21", "Kepulauan Riau", "Sumatera"),
    "kep. riau": ("21", "Kepulauan Riau", "Sumatera"),
    "dki jakarta": ("31", "DKI Jakarta", "Jawa"),
    "jawa barat": ("32", "Jawa Barat", "Jawa"),
    "jawa tengah": ("33", "Jawa Tengah", "Jawa"),
    "di yogyakarta": ("34", "DI Yogyakarta", "Jawa"),
    "d.i. yogyakarta": ("34", "DI Yogyakarta", "Jawa"),
    "daerah istimewa yogyakarta": ("34", "DI Yogyakarta", "Jawa"),
    "jawa timur": ("35", "Jawa Timur", "Jawa"),
    "banten": ("36", "Banten", "Jawa"),
    "bali": ("51", "Bali", "Bali & Nusa Tenggara"),
    "nusa tenggara barat": ("52", "Nusa Tenggara Barat", "Bali & Nusa Tenggara"),
    "nusa tenggara timur": ("53", "Nusa Tenggara Timur", "Bali & Nusa Tenggara"),
    "kalimantan barat": ("61", "Kalimantan Barat", "Kalimantan"),
    "kalimantan tengah": ("62", "Kalimantan Tengah", "Kalimantan"),
    "kalimantan selatan": ("63", "Kalimantan Selatan", "Kalimantan"),
    "kalimantan timur": ("64", "Kalimantan Timur", "Kalimantan"),
    "kalimantan utara": ("65", "Kalimantan Utara", "Kalimantan"),
    "sulawesi utara": ("71", "Sulawesi Utara", "Sulawesi"),
    "sulawesi tengah": ("72", "Sulawesi Tengah", "Sulawesi"),
    "sulawesi selatan": ("73", "Sulawesi Selatan", "Sulawesi"),
    "sulawesi tenggara": ("74", "Sulawesi Tenggara", "Sulawesi"),
    "gorontalo": ("75", "Gorontalo", "Sulawesi"),
    "sulawesi barat": ("76", "Sulawesi Barat", "Sulawesi"),
    "maluku": ("81", "Maluku", "Maluku & Papua"),
    "maluku utara": ("82", "Maluku Utara", "Maluku & Papua"),
    "papua barat": ("91", "Papua Barat", "Maluku & Papua"),
    "papua barat daya": ("92", "Papua Barat Daya", "Maluku & Papua"),
    "papua": ("94", "Papua", "Maluku & Papua"),
    "papua selatan": ("95", "Papua Selatan", "Maluku & Papua"),
    "papua tengah": ("96", "Papua Tengah", "Maluku & Papua"),
    "papua pegunungan": ("97", "Papua Pegunungan", "Maluku & Papua"),
}

NON_PROVINCE_LABELS = {"jawa", "luar jawa", "indonesia", "a. jawa", "b. luar jawa", "agregat (total)", "luar negeri"}

def normalize_province(raw_name):
    s = str(raw_name or "").strip()
    s = re.sub(r"^\d+\.\s*", "", s)
    s = re.sub(r"^[a-z]\.\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip()
    key = s.lower()
    if key in NON_PROVINCE_LABELS:
        return None, None, None, key
    if key in CANONICAL_38_PROVINCES:
        code, std_name, region = CANONICAL_38_PROVINCES[key]
        return code, std_name, region, None
    return None, s, None, "UNMATCHED"

def parse_ojk_month_iso(val):
    if isinstance(val, (datetime.datetime, datetime.date, pd.Timestamp)):
        return f"{val.year:04d}-{val.month:02d}-01"
    s = str(val).strip().lstrip("*").strip()
    m = re.match(r"^([A-Za-z]{3})-(\d{2})$", s)
    if m:
        mon_map = {
            "jan": 1, "feb": 2, "mar": 3, "apr": 4, "mei": 5, "may": 5,
            "jun": 6, "jul": 7, "agu": 8, "aug": 8, "sep": 9, "okt": 10,
            "oct": 10, "nov": 11, "des": 12, "dec": 12
        }
        mo = mon_map.get(m.group(1).lower())
        yr = 2000 + int(m.group(2))
        if mo:
            return f"{yr:04d}-{mo:02d}-01"
    dt = pd.to_datetime(s, errors="coerce")
    if pd.notna(dt):
        return f"{dt.year:04d}-{dt.month:02d}-01"
    return None

# 1. Extract and unpivot Sheet 9 from all 18 OJK XLSX files (newest release takes precedence on overlaps)
xlsx_records = []
unmatched_xlsx = set()
for fp in sorted((RAW_DIR / "ojk_xlsx").glob("*.xlsx")):
    release_tag = fp.stem.replace("ojk_lpbbti_", "")
    wb = openpyxl.load_workbook(fp, read_only=True, data_only=True)
    s9 = [s for s in wb.sheetnames if s.strip() == "9"][0]
    ws = wb[s9]
    rows_all = list(ws.iter_rows(min_row=1, max_row=60, values_only=True))
    wb.close()

    date_cols = []
    for col_idx, raw_dt in enumerate(rows_all[1]):
        if col_idx >= 2 and raw_dt is not None:
            iso_dt = parse_ojk_month_iso(raw_dt)
            if iso_dt:
                date_cols.append((col_idx, iso_dt))

    for row in rows_all[3:]:
        c1 = str(row[1]).strip() if len(row) > 1 and row[1] is not None else ""
        if not c1:
            continue
        if any(tok in c1.lower() for tok in ["jumlah", "total", "catatan", "sumber", "note", "*"]):
            continue
        prov_code, prov_name, region, flag = normalize_province(c1)
        if flag == "UNMATCHED":
            unmatched_xlsx.add(c1)
            continue
        if flag is not None:
            continue

        for col_idx, iso_dt in date_cols:
            if col_idx + 2 >= len(row):
                continue
            rek = pd.to_numeric(row[col_idx], errors="coerce")
            out_bn = pd.to_numeric(row[col_idx + 1], errors="coerce")
            twp_ratio = pd.to_numeric(row[col_idx + 2], errors="coerce")
            if pd.isna(out_bn) and pd.isna(twp_ratio):
                continue
            xlsx_records.append({
                "date": iso_dt,
                "prov_code": prov_code,
                "province": prov_name,
                "region": region,
                "xlsx_rekening_aktif": rek,
                "xlsx_outstanding_miliar": out_bn,
                "xlsx_twp90_pct": twp_ratio * 100.0 if pd.notna(twp_ratio) else np.nan,
                "xlsx_release": release_tag,
            })

df_xlsx_raw_long = pd.DataFrame(xlsx_records)
df_xlsx_clean = (
    df_xlsx_raw_long
    .sort_values(["date", "prov_code", "xlsx_release"])
    .drop_duplicates(subset=["date", "prov_code"], keep="last")
    .reset_index(drop=True)
)
print(f"step=xlsx_unpivot | raw_obs={len(df_xlsx_raw_long)} | dedup_obs={len(df_xlsx_clean)} | date_range={df_xlsx_clean['date'].min()}..{df_xlsx_clean['date'].max()} | n_months={df_xlsx_clean['date'].nunique()} | n_provs={df_xlsx_clean['prov_code'].nunique()} | unmatched={sorted(unmatched_xlsx)}")

# 2. Clean and aggregate SJKPublic provincial tables (Konvensional + Syariah)
sjk_dir = RAW_DIR / "ojk_sjkpublic"
excluded_log = {}

def prep_sjk_dati(df, table_tag, col_name="NamaDATI1"):
    out = df.copy()
    parsed = out[col_name].apply(normalize_province)
    out["prov_code"] = [p[0] for p in parsed]
    out["province"] = [p[1] for p in parsed]
    out["region"] = [p[2] for p in parsed]
    out["non_prov_flag"] = [p[3] for p in parsed]
    unm = out[out["non_prov_flag"] == "UNMATCHED"][col_name].unique().tolist()
    assert not unm, f"Unmatched DATI1 in {table_tag}: {unm}"
    excluded_counts = out.loc[out["non_prov_flag"].notna(), "non_prov_flag"].value_counts().to_dict()
    excluded_log[table_tag] = excluded_counts
    out = out[out["non_prov_flag"].isna()].copy()
    out["date"] = out["Bulan"].astype(str).str.strip() + "-01"
    return out

def build_quality_panel(df_q, tag, prefix):
    dq = prep_sjk_dati(df_q, tag, col_name="DATI1")
    piv = dq.pivot_table(
        index=["date", "prov_code", "province", "region"],
        columns="KualitasPembiayaan",
        values="OutstandingPerseoranganKualitasPerDati",
        aggfunc="sum",
    ).fillna(0.0)
    lancar = piv.get("001. Pendanaan Lancar", 0.0) / 1e9
    dpk = piv.get("002. Pendanaan Dalam Perhatian Khusus", 0.0) / 1e9
    kl = piv.get("003. Pendanaan Kurang Lancar", 0.0) / 1e9
    dirag = piv.get("004. Pendanaan Diragukan", 0.0) / 1e9
    macet = piv.get("005. Pendanaan Macet", 0.0) / 1e9
    total = lancar + dpk + kl + dirag + macet
    return pd.DataFrame({
        f"{prefix}_lancar_miliar": lancar,
        f"{prefix}_dpk_miliar": dpk,
        f"{prefix}_macet_miliar": macet,
        f"{prefix}_npl_early_miliar": kl + dirag + macet,
        f"{prefix}_outstanding_miliar": total,
    }).reset_index()

qk = build_quality_panel(pd.read_csv(sjk_dir / "1453_konv_borrower_kualitas_dati1_LPBBTI_PenerimaPinjamanKualitasPerDati.csv"), "1453_konv_qual", "konv")
qs = build_quality_panel(pd.read_csv(sjk_dir / "1454_syar_borrower_kualitas_dati1_LPBBTI_Syariah_PenerimaPinjamanKualitasPerDati.csv"), "1454_syar_qual", "syar")

df_sjk_qual = qk.merge(qs, on=["date", "prov_code", "province", "region"], how="outer").fillna(0.0)
df_sjk_qual["sjk_outstanding_miliar"] = df_sjk_qual["konv_outstanding_miliar"] + df_sjk_qual["syar_outstanding_miliar"]
df_sjk_qual["sjk_macet_miliar"] = df_sjk_qual["konv_macet_miliar"] + df_sjk_qual["syar_macet_miliar"]
df_sjk_qual["sjk_dpk_miliar"] = df_sjk_qual["konv_dpk_miliar"] + df_sjk_qual["syar_dpk_miliar"]
df_sjk_qual["sjk_npl_early_miliar"] = df_sjk_qual["konv_npl_early_miliar"] + df_sjk_qual["syar_npl_early_miliar"]
df_sjk_qual["sjk_twp90_pct"] = np.where(
    df_sjk_qual["sjk_outstanding_miliar"] > 0,
    (df_sjk_qual["sjk_macet_miliar"] / df_sjk_qual["sjk_outstanding_miliar"]) * 100.0,
    np.nan,
)
df_sjk_qual["sjk_early_delinq_pct"] = np.where(
    df_sjk_qual["sjk_outstanding_miliar"] > 0,
    ((df_sjk_qual["sjk_dpk_miliar"] + df_sjk_qual["sjk_npl_early_miliar"]) / df_sjk_qual["sjk_outstanding_miliar"]) * 100.0,
    np.nan,
)

def sum_konv_syar(fn_k, fn_s, tag_k, tag_s, val_col, out_col, div_scale=1.0, dati_col="NamaDATI1"):
    dk = prep_sjk_dati(pd.read_csv(sjk_dir / fn_k), tag_k, col_name=dati_col)
    ds = prep_sjk_dati(pd.read_csv(sjk_dir / fn_s), tag_s, col_name=dati_col)
    gk = dk.groupby(["date", "prov_code"], as_index=False)[val_col].sum().rename(columns={val_col: "vk"})
    gs = ds.groupby(["date", "prov_code"], as_index=False)[val_col].sum().rename(columns={val_col: "vs"})
    m = gk.merge(gs, on=["date", "prov_code"], how="outer").fillna(0.0)
    m[out_col] = (m["vk"] + m["vs"]) / div_scale
    return m[["date", "prov_code", out_col]]

# Note: verify disbursement/lender raw unit (Rupiah -> divide 1e9 for Miliar Rp)
df_bor_disb = sum_konv_syar("411_konv_borrower_dati1_LPBBTI_PenerimaPinjamanDATI1.csv", "427_syar_borrower_dati1_LPBBTI_Syariah_PenerimaPinjamanDATI1.csv", "411_konv_bor", "427_syar_bor", "JumlahDanaBrdsrknDATI1", "sjk_penyaluran_miliar", 1e9)
df_lnd_fund = sum_konv_syar("407_konv_lender_dati1_LPBBTI_PemberiPinjamanDATI1.csv", "423_syar_lender_dati1_LPBBTI_Syariah_PemberiPinjamanDATI1.csv", "407_konv_lnd", "423_syar_lnd", "JumlahDanaBrdsrknDATI1", "sjk_dana_lender_miliar", 1e9)
df_rek_act = sum_konv_syar("417_konv_rekening_dati1_LPBBTI_InfoKontrakUsahadanRekeningDATI1Penerima.csv", "433_syar_rekening_dati1_LPBBTI_Syariah_InfoKontrakUsahadanRekeningDATI1Penerima.csv", "417_konv_rek", "433_syar_rek", "JumlahRekeningPenerimaPinjaman", "sjk_rekening_aktif", 1.0, dati_col="AkunPenerimaDATI1")
df_nat_sh = sum_konv_syar("144_konv_twp90_dati1_LPBBTI_TWP90_DATI1.csv", "312_syar_twp90_dati1_LPBBTI_Syariah_TWP90_DATI1.csv", "144_konv_twp", "312_syar_twp", "TWP90", "sjk_twp90_nat_share_pct", 1.0)

# Extract monthly foreign lender funding from 407+423 as macro context
fk = pd.read_csv(sjk_dir / "407_konv_lender_dati1_LPBBTI_PemberiPinjamanDATI1.csv")
fs = pd.read_csv(sjk_dir / "423_syar_lender_dati1_LPBBTI_Syariah_PemberiPinjamanDATI1.csv")
fk_f = fk[fk["NamaDATI1"].str.contains("Luar Negeri", case=False, na=False)].assign(date=lambda d: d["Bulan"].astype(str) + "-01").groupby("date", as_index=False)["JumlahDanaBrdsrknDATI1"].sum()
fs_f = fs[fs["NamaDATI1"].str.contains("Luar Negeri", case=False, na=False)].assign(date=lambda d: d["Bulan"].astype(str) + "-01").groupby("date", as_index=False)["JumlahDanaBrdsrknDATI1"].sum()
df_foreign_lender = fk_f.merge(fs_f, on="date", how="outer", suffixes=("_k", "_s")).fillna(0.0)
df_foreign_lender["nat_lender_luar_negeri_miliar"] = (df_foreign_lender["JumlahDanaBrdsrknDATI1_k"] + df_foreign_lender["JumlahDanaBrdsrknDATI1_s"]) / 1e9

df_sjk = df_sjk_qual.merge(df_bor_disb, on=["date", "prov_code"], how="left")
df_sjk = df_sjk.merge(df_lnd_fund, on=["date", "prov_code"], how="left")
df_sjk = df_sjk.merge(df_rek_act, on=["date", "prov_code"], how="left")
df_sjk = df_sjk.merge(df_nat_sh, on=["date", "prov_code"], how="left")

print("\nstep=non_province_exclusions_audit")
for tag, cnts in excluded_log.items():
    print(f"  {tag}: {cnts}")

# 3. Verify overlap parity (2024-07..2025-12) between XLSX Sheet 9 and SJKPublic
df_ojk = df_xlsx_clean.merge(
    df_sjk,
    on=["date", "prov_code"],
    how="outer",
    suffixes=("_xlsx", "_sjk"),
)
df_ojk["province"] = df_ojk["province_sjk"].combine_first(df_ojk["province_xlsx"])
df_ojk["region"] = df_ojk["region_sjk"].combine_first(df_ojk["region_xlsx"])
df_ojk = df_ojk.merge(df_foreign_lender[["date", "nat_lender_luar_negeri_miliar"]], on="date", how="left")

overlap = df_ojk.dropna(subset=["xlsx_twp90_pct", "sjk_twp90_pct"]).copy()
overlap["abs_diff_twp90_pp"] = (overlap["xlsx_twp90_pct"] - overlap["sjk_twp90_pct"]).abs()
overlap["abs_diff_out_miliar"] = (overlap["xlsx_outstanding_miliar"] - overlap["sjk_outstanding_miliar"]).abs()

print(f"\nstep=overlap_parity_audit | overlap_rows={len(overlap)} | overlap_months={overlap['date'].min()}..{overlap['date'].max()}")
print(f"  max_abs_diff_twp90_pp={overlap['abs_diff_twp90_pp'].max():.6f} | median_abs_diff_twp90_pp={overlap['abs_diff_twp90_pp'].median():.6f}")
print(f"  max_abs_diff_out_miliar={overlap['abs_diff_out_miliar'].max():.6f} | median_abs_diff_out_miliar={overlap['abs_diff_out_miliar'].median():.6f}")

# 4. Construct unified OJK provincial series (prefer SJKPublic where available, backfill 2022-01..2024-06 from XLSX)
df_ojk["twp90_pct"] = df_ojk["sjk_twp90_pct"].combine_first(df_ojk["xlsx_twp90_pct"])
df_ojk["outstanding_miliar"] = df_ojk["sjk_outstanding_miliar"].combine_first(df_ojk["xlsx_outstanding_miliar"])
df_ojk["rekening_aktif"] = df_ojk["sjk_rekening_aktif"].combine_first(df_ojk["xlsx_rekening_aktif"])
df_ojk["data_source_ojk"] = np.where(df_ojk["sjk_twp90_pct"].notna(), "SJKPublic+XLSX", "XLSX_Historical")

rows_before_filter = len(df_ojk)
df_ojk = df_ojk[df_ojk["twp90_pct"].notna()].sort_values(["date", "prov_code"]).reset_index(drop=True)
assert df_ojk.duplicated(subset=["date", "prov_code"]).sum() == 0, "Duplicate (date, prov_code) found!"

ojk_out = CLEAN_DIR / "ojk_provincial_panel_clean.csv"
df_ojk.to_csv(ojk_out, index=False)

print(f"\nstep=ojk_panel_saved | file={ojk_out.name} | rows_pre_filter={rows_before_filter} | final_rows={len(df_ojk)} | dates={df_ojk['date'].min()}..{df_ojk['date'].max()} ({df_ojk['date'].nunique()} months) | n_provs={df_ojk['prov_code'].nunique()}")
print("  twp90_pct_describe=", df_ojk["twp90_pct"].describe().round(4).to_dict())
print("  breach_5pct_count=", int((df_ojk["twp90_pct"] >= 5.0).sum()), f"({(df_ojk['twp90_pct'] >= 5.0).mean()*100:.2f}%)")
print("  sample_sjk_units_2026_04:")
print(df_ojk[df_ojk["date"] == "2026-04-01"].sort_values("outstanding_miliar", ascending=False).head(5)[["date", "province", "twp90_pct", "outstanding_miliar", "sjk_penyaluran_miliar", "sjk_dana_lender_miliar", "rekening_aktif"]].to_string(index=False))


step=xlsx_unpivot | raw_obs=8702 | dedup_obs=1704 | date_range=2022-01-01..2025-12-01 | n_months=48 | n_provs=38 | unmatched=[]



step=non_province_exclusions_audit
  1453_konv_qual: {}
  1454_syar_qual: {}
  411_konv_bor: {'jawa': 22, 'luar jawa': 22, 'agregat (total)': 22}
  427_syar_bor: {'jawa': 22, 'luar jawa': 22, 'agregat (total)': 22}
  407_konv_lnd: {'jawa': 22, 'luar jawa': 22, 'luar negeri': 22, 'agregat (total)': 22}
  423_syar_lnd: {'jawa': 22, 'luar jawa': 22, 'luar negeri': 22, 'agregat (total)': 22}
  417_konv_rek: {'jawa': 26, 'luar jawa': 26, 'agregat (total)': 26}
  433_syar_rek: {'jawa': 26, 'luar jawa': 26, 'agregat (total)': 26}
  144_konv_twp: {'luar jawa': 22, 'jawa': 22}
  312_syar_twp: {'luar jawa': 22, 'jawa': 22}

step=overlap_parity_audit | overlap_rows=684 | overlap_months=2024-07-01..2025-12-01
  max_abs_diff_twp90_pp=0.936980 | median_abs_diff_twp90_pp=0.000000
  max_abs_diff_out_miliar=422.708899 | median_abs_diff_out_miliar=0.000000



step=ojk_panel_saved | file=ojk_provincial_panel_clean.csv | rows_pre_filter=1856 | final_rows=1856 | dates=2022-01-01..2026-04-01 (52 months) | n_provs=38
  twp90_pct_describe= {'count': 1856.0, 'mean': 2.0085, 'std': 1.0052, 'min': 0.032, '25%': 1.3805, '50%': 1.8248, '75%': 2.4363, 'max': 11.5793}
  breach_5pct_count= 33 (1.78%)
  sample_sjk_units_2026_04:
      date    province  twp90_pct  outstanding_miliar  sjk_penyaluran_miliar  sjk_dana_lender_miliar  rekening_aktif
2026-04-01  Jawa Barat   3.341740        24674.328930            7723.823083              158.925770       4726802.0
2026-04-01 DKI Jakarta   3.468331        13931.147298            4214.089627            22243.479936       2042036.0
2026-04-01  Jawa Timur   5.858248        11407.337143            3290.950567              202.559462       1928107.0
2026-04-01 Jawa Tengah   3.445968         8216.279479            2498.274676               28.808088       1537335.0
2026-04-01      Banten   2.771701         7130.21556

In [3]:
# Cell 3 - Clean and unify all 11 BPS and UMP raw CSVs into canonical 38-province x year dataset
bps_dir = RAW_DIR / "bps"

BPS_EXTRA_ALIASES = {
    "d i yogyakarta": ("34", "DI Yogyakarta", "Jawa"),
    "daerah khusus ibukota jakarta": ("31", "DKI Jakarta", "Jawa"),
}
BPS_PROV_MAP = {**CANONICAL_38_PROVINCES, **BPS_EXTRA_ALIASES}
BPS_NON_PROVINCE = NON_PROVINCE_LABELS | {"rata-rata", "total", "jumlah"}

def normalize_bps_province(raw_name):
    s = str(raw_name or "").strip()
    s = re.sub(r"<[^>]+>", "", s)
    s = re.sub(r"^\d+\.?\s*", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    key = s.lower()
    if key in BPS_NON_PROVINCE:
        return None, None, None, key
    if key in BPS_PROV_MAP:
        code, std_name, region = BPS_PROV_MAP[key]
        return code, std_name, region, None
    return None, s, None, "UNMATCHED"

def parse_id_number(val, decimal_comma=True):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    s = re.sub(r"[\*\s]+$", "", s).strip()
    if s in {"", "...", "-", "–", "NA", "nan", "None"}:
        return np.nan
    if decimal_comma:
        if "," in s:
            s = s.replace(".", "").replace(",", ".")
        elif re.fullmatch(r"\d{1,3}(\.\d{3})+", s):
            s = s.replace(".", "")
    return pd.to_numeric(s, errors="coerce")

bps_drop_log = {}

def load_and_clean_bps_file(fn, force_str=False, filter_prov_code_mod100=False):
    dtype_kw = {"nilai_raw": str} if force_str else {}
    df = pd.read_csv(bps_dir / fn, dtype=dtype_kw)
    raw_n = len(df)
    if filter_prov_code_mod100:
        keep_mask = df["kode_prov"].astype(int) % 100 == 0
        bps_drop_log[f"{fn}_kabkota_filtered"] = int((~keep_mask).sum())
        df = df[keep_mask].copy()
    parsed = df["provinsi_raw"].apply(normalize_bps_province)
    df["prov_code"] = [p[0] for p in parsed]
    df["province"] = [p[1] for p in parsed]
    df["region"] = [p[2] for p in parsed]
    df["non_prov_flag"] = [p[3] for p in parsed]
    unm = df[df["non_prov_flag"] == "UNMATCHED"]["provinsi_raw"].unique().tolist()
    assert not unm, f"Unmatched BPS province in {fn}: {unm}"
    bps_drop_log[f"{fn}_agg_filtered"] = int(df["non_prov_flag"].notna().sum())
    df = df[df["non_prov_flag"].isna()].copy()
    df["value"] = df["nilai_raw"].apply(lambda v: parse_id_number(v, decimal_comma=force_str))
    df["tahun"] = df["tahun"].astype(int)
    print(f"  cleaned={fn:30s} | raw={raw_n:4d} -> prov_rows={len(df):4d} | years={sorted(df['tahun'].unique())} | nulls={int(df['value'].isna().sum())}")
    return df

print("step=bps_cleaning_audit")
df_tpt = load_and_clean_bps_file("bps_tpt_raw.csv", force_str=False)
df_inf = load_and_clean_bps_file("bps_informal_raw.csv", force_str=False)
df_upah_j = load_and_clean_bps_file("bps_upah_jam_raw.csv", force_str=False)
df_upah_b = load_and_clean_bps_file("bps_upah_bersih_raw.csv", force_str=False)
df_ipm = load_and_clean_bps_file("bps_ipm_raw.csv", force_str=False, filter_prov_code_mod100=True)
df_ump_bps = load_and_clean_bps_file("bps_ump_raw.csv", force_str=False)
df_miskin = load_and_clean_bps_file("bps_kemiskinan_raw.csv", force_str=True)
df_pengel = load_and_clean_bps_file("bps_pengeluaran_raw.csv", force_str=True)
df_pend = load_and_clean_bps_file("bps_penduduk_raw.csv", force_str=True)
df_pdrb = load_and_clean_bps_file("bps_pdrb_perkapita_raw.csv", force_str=True)

# Clean Refnusa UMP (2022-2026) and verify parity against BPS UMP (2023-2026)
df_ump_ref = pd.read_csv(bps_dir / "refnusa_ump_1997_2026_raw.csv")
df_ump_ref = df_ump_ref[df_ump_ref["year"].between(2022, 2026)].copy()
parsed_ref = df_ump_ref["province_name"].apply(normalize_bps_province)
df_ump_ref["prov_code"] = [p[0] for p in parsed_ref]
df_ump_ref["province"] = [p[1] for p in parsed_ref]
df_ump_ref["non_prov_flag"] = [p[3] for p in parsed_ref]
unm_ref = df_ump_ref[df_ump_ref["non_prov_flag"] == "UNMATCHED"]["province_name"].unique().tolist()
assert not unm_ref, f"Unmatched province in Refnusa UMP: {unm_ref}"
df_ump_ref = df_ump_ref[df_ump_ref["non_prov_flag"].isna()].rename(columns={"year": "tahun", "amount_idr": "ump_ref_idr"})

ump_cmp = df_ump_bps[["tahun", "prov_code", "province", "value"]].rename(columns={"value": "ump_bps_idr"}).merge(
    df_ump_ref[["tahun", "prov_code", "ump_ref_idr"]], on=["tahun", "prov_code"], how="outer"
)
ump_overlap = ump_cmp.dropna(subset=["ump_bps_idr", "ump_ref_idr"]).copy()
ump_overlap["abs_diff_idr"] = (ump_overlap["ump_bps_idr"] - ump_overlap["ump_ref_idr"]).abs()
print(f"\nstep=ump_parity_check | overlap_rows={len(ump_overlap)} | exact_matches={(ump_overlap['abs_diff_idr'] < 1.0).sum()} | max_diff_idr={ump_overlap['abs_diff_idr'].max():.2f}")

ump_cmp["ump_idr"] = ump_cmp["ump_bps_idr"].combine_first(ump_cmp["ump_ref_idr"])
ump_cmp["ump_source"] = np.where(ump_cmp["ump_bps_idr"].notna(), "BPS_2824", "Refnusa_Gub")

# Pivot each BPS indicator to (tahun, prov_code) grain
piv_tpt = df_tpt.pivot_table(index=["tahun", "prov_code"], columns="periode", values="value", aggfunc="mean").reset_index()
piv_tpt = piv_tpt.rename(columns={"Februari": "tpt_feb_pct", "Agustus": "tpt_aug_pct"})
piv_tpt["tpt_mean_pct"] = piv_tpt[["tpt_feb_pct", "tpt_aug_pct"]].mean(axis=1)

piv_inf = df_inf.groupby(["tahun", "prov_code"], as_index=False)["value"].mean().rename(columns={"value": "informal_worker_pct"})
piv_uj = df_upah_j.groupby(["tahun", "prov_code"], as_index=False)["value"].mean().rename(columns={"value": "upah_jam_idr"})
piv_ub = (
    df_upah_b[df_upah_b["turunan_var"].str.strip().str.lower() == "jumlah"]
    .groupby(["tahun", "prov_code"], as_index=False)["value"]
    .mean()
    .rename(columns={"value": "upah_bersih_bulan_idr"})
)
piv_ipm = df_ipm.groupby(["tahun", "prov_code"], as_index=False)["value"].mean().rename(columns={"value": "ipm_score"})

piv_mk = df_miskin.pivot_table(index=["tahun", "prov_code"], columns="variabel", values="value", aggfunc="mean").reset_index()
mk_panel = pd.DataFrame({
    "tahun": piv_mk["tahun"],
    "prov_code": piv_mk["prov_code"],
    "garis_kemiskinan_idr": piv_mk[["Garis Kemiskinan - Maret", "Garis Kemiskinan - September"]].mean(axis=1),
    "kemiskinan_pct": piv_mk[["Persentase Penduduk Miskin - Maret", "Persentase Penduduk Miskin - September"]].mean(axis=1),
    "penduduk_miskin_ribu": piv_mk[["Jumlah Penduduk Miskin - Maret (ribu)", "Jumlah Penduduk Miskin - September (ribu)"]].mean(axis=1),
})

piv_pg = df_pengel.pivot_table(index=["tahun", "prov_code"], columns="variabel", values="value", aggfunc="mean").reset_index()
pg_panel = pd.DataFrame({
    "tahun": piv_pg["tahun"],
    "prov_code": piv_pg["prov_code"],
    "pengeluaran_makanan_idr": piv_pg["Rata-rata Pengeluaran per Kapita Sebulan di Perkotaan dan Perdesaan - Makanan"],
    "pengeluaran_nonmakanan_idr": piv_pg["Rata-rata Pengeluaran per Kapita Sebulan di Perkotaan dan Perdesaan - Bukan Makanan"],
    "pengeluaran_total_idr": piv_pg["Rata-rata Pengeluaran per Kapita Sebulan di Perkotaan dan Perdesaan - Jumlah"],
})
pg_panel["rasio_pengeluaran_nonmakanan"] = pg_panel["pengeluaran_nonmakanan_idr"] / pg_panel["pengeluaran_total_idr"]

piv_pd = df_pend.pivot_table(index=["tahun", "prov_code"], columns="variabel", values="value", aggfunc="mean").reset_index()
pd_panel = pd.DataFrame({
    "tahun": piv_pd["tahun"],
    "prov_code": piv_pd["prov_code"],
    "penduduk_ribu": piv_pd["Jumlah Penduduk"],
    "laju_pertumbuhan_penduduk_pct": piv_pd["Laju Pertumbuhan Penduduk per Tahun"],
    "kepadatan_penduduk_km2": piv_pd["Kepadatan Penduduk per km persegi"],
})

piv_pdrb = df_pdrb.groupby(["tahun", "prov_code"], as_index=False)["value"].mean().rename(columns={"value": "pdrb_perkapita_ribu_idr"})

prov_meta = pd.DataFrame(
    list({v[0]: (v[0], v[1], v[2]) for v in CANONICAL_38_PROVINCES.values()}.values()),
    columns=["prov_code", "province", "region"],
)
grid = pd.MultiIndex.from_product([[2022, 2023, 2024, 2025, 2026], sorted(prov_meta["prov_code"])], names=["tahun", "prov_code"]).to_frame(index=False)
df_bps_clean = grid.merge(prov_meta, on="prov_code", how="left")

for sub_df in [
    ump_cmp[["tahun", "prov_code", "ump_idr", "ump_source"]],
    piv_tpt[["tahun", "prov_code", "tpt_feb_pct", "tpt_aug_pct", "tpt_mean_pct"]],
    piv_inf, piv_uj, piv_ub, piv_ipm, mk_panel, pg_panel, pd_panel, piv_pdrb
]:
    df_bps_clean = df_bps_clean.merge(sub_df, on=["tahun", "prov_code"], how="left")

assert df_bps_clean.duplicated(subset=["tahun", "prov_code"]).sum() == 0, "Duplicate (tahun, prov_code) in BPS clean!"
bps_out = CLEAN_DIR / "bps_provincial_clean.csv"
df_bps_clean.to_csv(bps_out, index=False)

print(f"\nstep=bps_clean_saved | file={bps_out.name} | shape={df_bps_clean.shape} | drop_log={bps_drop_log}")
print("  nulls_by_year:")
null_yr = df_bps_clean.groupby("tahun")[["ump_idr", "tpt_mean_pct", "informal_worker_pct", "upah_bersih_bulan_idr", "ipm_score", "kemiskinan_pct", "pengeluaran_total_idr", "penduduk_ribu", "pdrb_perkapita_ribu_idr"]].apply(lambda g: g.isna().sum())
print(null_yr.to_string())
print("\n  sample_2025_top5_pdrb:")
print(df_bps_clean[df_bps_clean["tahun"] == 2025].sort_values("pdrb_perkapita_ribu_idr", ascending=False).head(5)[["province", "ump_idr", "tpt_mean_pct", "kemiskinan_pct", "pengeluaran_total_idr", "pdrb_perkapita_ribu_idr"]].to_string(index=False))


step=bps_cleaning_audit
  cleaned=bps_tpt_raw.csv                | raw= 304 -> prov_rows= 296 | years=[np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)] | nulls=0
  cleaned=bps_informal_raw.csv           | raw= 113 -> prov_rows= 110 | years=[np.int64(2023), np.int64(2024), np.int64(2025)] | nulls=0
  cleaned=bps_upah_jam_raw.csv           | raw= 113 -> prov_rows= 110 | years=[np.int64(2023), np.int64(2024), np.int64(2025)] | nulls=0


  cleaned=bps_upah_bersih_raw.csv        | raw=1733 -> prov_rows=1688 | years=[np.int64(2024), np.int64(2025), np.int64(2026)] | nulls=0
  cleaned=bps_ipm_raw.csv                | raw=1132 -> prov_rows=  76 | years=[np.int64(2023), np.int64(2024)] | nulls=0
  cleaned=bps_ump_raw.csv                | raw= 147 -> prov_rows= 147 | years=[np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)] | nulls=0
  cleaned=bps_kemiskinan_raw.csv         | raw= 567 -> prov_rows= 552 | years=[np.int64(2023), np.int64(2024), np.int64(2025)] | nulls=0
  cleaned=bps_pengeluaran_raw.csv        | raw= 339 -> prov_rows= 330 | years=[np.int64(2023), np.int64(2024), np.int64(2025)] | nulls=0


  cleaned=bps_penduduk_raw.csv           | raw= 780 -> prov_rows= 760 | years=[np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)] | nulls=20
  cleaned=bps_pdrb_perkapita_raw.csv     | raw= 114 -> prov_rows= 114 | years=[np.int64(2023), np.int64(2024), np.int64(2025)] | nulls=0

step=ump_parity_check | overlap_rows=146 | exact_matches=146 | max_diff_idr=0.90



step=bps_clean_saved | file=bps_provincial_clean.csv | shape=(190, 24) | drop_log={'bps_tpt_raw.csv_agg_filtered': 8, 'bps_informal_raw.csv_agg_filtered': 3, 'bps_upah_jam_raw.csv_agg_filtered': 3, 'bps_upah_bersih_raw.csv_agg_filtered': 45, 'bps_ipm_raw.csv_kabkota_filtered': 1056, 'bps_ipm_raw.csv_agg_filtered': 0, 'bps_ump_raw.csv_agg_filtered': 0, 'bps_kemiskinan_raw.csv_agg_filtered': 15, 'bps_pengeluaran_raw.csv_agg_filtered': 9, 'bps_penduduk_raw.csv_agg_filtered': 20, 'bps_pdrb_perkapita_raw.csv_agg_filtered': 0}
  nulls_by_year:
       ump_idr  tpt_mean_pct  informal_worker_pct  upah_bersih_bulan_idr  ipm_score  kemiskinan_pct  pengeluaran_total_idr  penduduk_ribu  pdrb_perkapita_ribu_idr
tahun                                                                                                                                                             
2022         4            38                   38                     38         38              38                     38       

In [4]:
# Cell 4 - Clean OJK Direktori, Play Store apps/reviews (drop 3 false matches), and Satgas PASTI
import ast
from urllib.parse import urlparse

MONTH_MAP_ID = {
    "januari": 1, "jan": 1,
    "februari": 2, "feb": 2,
    "maret": 3, "mar": 3,
    "april": 4, "apr": 4,
    "mei": 5, "may": 5,
    "juni": 6, "jun": 6,
    "juli": 7, "jul": 7,
    "agustus": 8, "agu": 8, "aug": 8,
    "september": 9, "sep": 9,
    "oktober": 10, "okt": 10, "oct": 10,
    "november": 11, "nov": 11,
    "desember": 12, "des": 12, "dec": 12,
}

def parse_id_date_iso(val):
    if pd.isna(val):
        return None
    s = str(val).replace("\xa0", " ").strip()
    s = re.sub(r"[-/]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    m = re.match(r"^(\d{1,2})\s+([A-Za-z]+)\s+(\d{2,4})$", s)
    if m:
        day = int(m.group(1))
        mo = MONTH_MAP_ID.get(m.group(2).lower())
        yr = int(m.group(3))
        if yr < 100:
            yr += 2000
        if mo:
            return f"{yr:04d}-{mo:02d}-{day:02d}"
    dt = pd.to_datetime(val, errors="coerce")
    return dt.strftime("%Y-%m-%d") if pd.notna(dt) else None

# 1. Clean OJK Direktori LPBBTI Mei 2026
df_dir_c = pd.read_csv(RAW_DIR / "ojk_directory" / "ojk_direktori_lpbbti_mei2026_raw.csv")
df_dir_c["nama_perusahaan"] = df_dir_c["nama_perusahaan"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df_dir_c["nama_sistem_elektronik"] = df_dir_c["nama_sistem_elektronik"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df_dir_c["tanggal_izin"] = df_dir_c["tanggal_izin_raw"].apply(parse_id_date_iso)
assert df_dir_c["tanggal_izin"].isna().sum() == 0, "Failed parsing tanggal_izin in directory!"

os_before = df_dir_c["sistem_operasi"].value_counts().to_dict()
df_dir_c["sistem_operasi_clean"] = (
    df_dir_c["sistem_operasi"]
    .astype(str)
    .str.strip()
    .replace({"Andorid": "Android", "-": "No Mobile App Listed"})
)
df_dir_c["has_android_ojk"] = df_dir_c["sistem_operasi_clean"].str.contains("Android", case=False, na=False).astype(int)
df_dir_c["has_ios_ojk"] = df_dir_c["sistem_operasi_clean"].str.contains("iOS", case=False, na=False).astype(int)
print(f"step=clean_ojk_directory | rows={len(df_dir_c)} | date_nulls={df_dir_c['tanggal_izin'].isna().sum()} | izin_span={df_dir_c['tanggal_izin'].min()}..{df_dir_c['tanggal_izin'].max()}")
print(f"  os_before={os_before} -> os_after={df_dir_c['sistem_operasi_clean'].value_counts().to_dict()}")

# 2. Clean Play Store Apps: explicitly drop 3 verified false-positive matches
FALSE_MATCH_OJK_IDS = {12, 27, 88}  # Ammana->ALAMI, OVO Finansial->Singa, KawanCicil->CICIL
df_apps_raw = pd.read_csv(RAW_DIR / "playstore" / "playstore_pindar_apps_raw.csv")
n_apps_raw = len(df_apps_raw)
df_apps_c = df_apps_raw[~df_apps_raw["no_ojk"].isin(FALSE_MATCH_OJK_IDS)].copy()
assert df_apps_c["appId"].duplicated().sum() == 0, "Remaining duplicate appIds in Play Store apps!"

df_apps_c["app_released_date"] = df_apps_c["released"].apply(parse_id_date_iso)
df_apps_c["app_updated_date"] = pd.to_datetime(df_apps_c["updated"], unit="s", errors="coerce").dt.strftime("%Y-%m-%d")

def parse_hist(val):
    try:
        arr = ast.literal_eval(str(val))
        if isinstance(arr, list) and len(arr) == 5:
            return arr
    except Exception:
        pass
    return [np.nan] * 5

hist_vals = df_apps_c["histogram"].apply(parse_hist).tolist()
for star_idx in range(5):
    df_apps_c[f"star_{star_idx + 1}_count"] = [h[star_idx] for h in hist_vals]
tot_stars = df_apps_c[[f"star_{i}_count" for i in range(1, 6)]].sum(axis=1)
df_apps_c["star_1_share_pct"] = np.where(tot_stars > 0, (df_apps_c["star_1_count"] / tot_stars) * 100.0, np.nan)
df_apps_c["star_5_share_pct"] = np.where(tot_stars > 0, (df_apps_c["star_5_count"] / tot_stars) * 100.0, np.nan)

df_dir_merged = df_dir_c.merge(
    df_apps_c[["no_ojk", "appId", "title", "developer", "score", "ratings", "reviews_total", "realInstalls", "app_released_date", "app_updated_date", "star_1_share_pct", "star_5_share_pct", "match_rule"]],
    left_on="no",
    right_on="no_ojk",
    how="left",
).drop(columns=["no_ojk"])
dir_out = CLEAN_DIR / "ojk_direktori_playstore_clean.csv"
df_dir_merged.to_csv(dir_out, index=False)
print(f"step=clean_playstore_apps | raw={n_apps_raw} -> verified={len(df_apps_c)} (dropped_false_matches={n_apps_raw - len(df_apps_c)}) | merged_dir_rows={len(df_dir_merged)}")

# 3. Clean Play Store Reviews: drop reviews tied to the 3 false matches and engineer complaint flags
df_rev_raw = pd.read_csv(RAW_DIR / "playstore" / "playstore_pindar_reviews_raw.csv")
n_rev_raw = len(df_rev_raw)
df_rev_c = df_rev_raw[~df_rev_raw["no_ojk"].isin(FALSE_MATCH_OJK_IDS)].copy()
df_rev_c = df_rev_c.drop_duplicates(subset=["reviewId"]).reset_index(drop=True)
assert df_rev_c["reviewId"].duplicated().sum() == 0, "Duplicate reviewIds remain!"

df_rev_c["review_text_clean"] = df_rev_c["content"].astype(str).str.replace("\xa0", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip()
df_rev_c["review_date"] = pd.to_datetime(df_rev_c["at"], errors="coerce").dt.strftime("%Y-%m-%d")
df_rev_c["review_month"] = pd.to_datetime(df_rev_c["at"], errors="coerce").dt.strftime("%Y-%m-01")
df_rev_c["has_developer_reply"] = df_rev_c["replyContent"].notna().astype(int)
df_rev_c["is_negative_review"] = (df_rev_c["score"] <= 2).astype(int)

txt_low = df_rev_c["review_text_clean"].str.lower()
df_rev_c["kw_collection_terror"] = txt_low.str.contains(r"\b(?:dc|collector|kolektor|teror|ancam|sebar data|sebar|kasar|tagih|penagih|telpon|telepon)\b", regex=True).astype(int)
df_rev_c["kw_cost_interest"] = txt_low.str.contains(r"\b(?:bunga|denda|admin|potongan|tenor|mahal|cekik|riba|cicilan)\b", regex=True).astype(int)
df_rev_c["kw_app_tech_issue"] = txt_low.str.contains(r"\b(?:error|eror|bug|gagal|login|loading|lemot|otp|verifikasi|ktp|wajah|sistem|jaringan)\b", regex=True).astype(int)
df_rev_c["kw_rejection_limit"] = txt_low.str.contains(r"\b(?:ditolak|tolak|cair|pencairan|limit|acc)\b", regex=True).astype(int)

rev_out = CLEAN_DIR / "playstore_reviews_clean.csv"
df_rev_c.to_csv(rev_out, index=False)
print(f"step=clean_playstore_reviews | raw={n_rev_raw} -> clean={len(df_rev_c)} (dropped_dups_false={n_rev_raw - len(df_rev_c)}) | apps={df_rev_c['appId'].nunique()} | span={df_rev_c['review_date'].min()}..{df_rev_c['review_date'].max()}")
print("  review_theme_rates_pct=", {c: round(float(df_rev_c[c].mean() * 100.0), 2) for c in ["is_negative_review", "kw_collection_terror", "kw_cost_interest", "kw_app_tech_issue", "kw_rejection_limit", "has_developer_reply"]})

# 4. Clean Satgas PASTI Press Releases and Illegal Pinjol List
df_sp_raw = pd.read_csv(RAW_DIR / "playstore" / "satgas_pasti_siaran_pers_raw.csv")
df_sp_c = df_sp_raw.copy()
df_sp_c["title_clean"] = df_sp_c["title"].astype(str).str.replace("\xa0", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip()
df_sp_c["summary_clean"] = df_sp_c["summary"].fillna("").astype(str).str.replace("\xa0", " ", regex=False).str.replace(r"\s+", " ", regex=True).str.strip()
df_sp_c["press_date"] = df_sp_c["date_raw"].apply(parse_id_date_iso)
df_sp_c["press_month"] = pd.to_datetime(df_sp_c["press_date"], errors="coerce").dt.strftime("%Y-%m-01")
assert df_sp_c["press_date"].isna().sum() == 0, "Failed parsing press_date in Satgas PASTI!"

def extract_blocked_count(row):
    text = f"{row['title_clean']} {row['summary_clean']}"
    m = re.search(r"(\d[\d.]*)\s+(?:aktivitas|entitas|pinjaman online|pinjol)", text, flags=re.IGNORECASE)
    if m:
        num = m.group(1).replace(".", "")
        if num.isdigit():
            return int(num)
    return np.nan

df_sp_c["entities_blocked_mentioned"] = df_sp_c.apply(extract_blocked_count, axis=1)
df_sp_c["mentions_pinjol_or_scam"] = (df_sp_c["title_clean"] + " " + df_sp_c["summary_clean"]).str.contains(r"pinjol|pinjaman|scam|iasc|pindar|ilegal", case=False, regex=True).astype(int)

sp_out = CLEAN_DIR / "satgas_pasti_press_clean.csv"
df_sp_c.to_csv(sp_out, index=False)

df_ill_raw = pd.read_csv(RAW_DIR / "playstore" / "satgas_pasti_pinjol_ilegal_okt2026_raw.csv")
df_ill_c = df_ill_raw.copy()
df_ill_c["url_clean"] = df_ill_c["url"].astype(str).str.strip().str.lower()
df_ill_c["domain"] = df_ill_c["url_clean"].apply(lambda u: urlparse(u if "://" in u else "http://" + u).netloc)
df_ill_c["platform_channel"] = np.where(
    df_ill_c["domain"].str.contains("blogspot|wordpress|sites.google|linktr|wix", na=False), "Blog/FreeHost",
    np.where(df_ill_c["url_clean"].str.contains(r"\.apk|apkpure|softonic|uptodown", regex=True, na=False), "ThirdParty_APK", "Standalone_Web/App")
)
df_ill_c["nama_entitas_clean"] = df_ill_c["nama_entitas"].fillna("UNKNOWN_ENTITY").astype(str).str.strip()
ill_out = CLEAN_DIR / "satgas_pasti_illegal_pinjol_okt2026_clean.csv"
df_ill_c.to_csv(ill_out, index=False)

print(f"step=clean_satgas_pasti | press_rows={len(df_sp_c)} | span={df_sp_c['press_date'].min()}..{df_sp_c['press_date'].max()} | with_blocked_count={int(df_sp_c['entities_blocked_mentioned'].notna().sum())}")
print(f"  illegal_pinjol_okt2026={len(df_ill_c)} | channels={df_ill_c['platform_channel'].value_counts().to_dict()}")


step=clean_ojk_directory | rows=94 | date_nulls=0 | izin_span=2017-07-06..2021-12-23
  os_before={'Android': 37, 'Android & iOS': 32, '-': 24, 'Andorid': 1} -> os_after={'Android': 38, 'Android & iOS': 32, 'No Mobile App Listed': 24}


step=clean_playstore_apps | raw=72 -> verified=69 (dropped_false_matches=3) | merged_dir_rows=94


step=clean_playstore_reviews | raw=4061 -> clean=3881 (dropped_dups_false=180) | apps=66 | span=2021-10-12..2026-11-10
  review_theme_rates_pct= {'is_negative_review': 35.84, 'kw_collection_terror': 1.49, 'kw_cost_interest': 4.64, 'kw_app_tech_issue': 7.81, 'kw_rejection_limit': 16.13, 'has_developer_reply': 57.49}
step=clean_satgas_pasti | press_rows=79 | span=2016-11-01..2025-11-15 | with_blocked_count=53
  illegal_pinjol_okt2026=330 | channels={'Standalone_Web/App': 245, 'ThirdParty_APK': 83, 'Blog/FreeHost': 2}


In [5]:
# Cell 5 - Causal Master Panel Integration, Missingness Audit, Outlier Detection & Justification
from sklearn.ensemble import IsolationForest

ojk = pd.read_csv(CLEAN_DIR / "ojk_provincial_panel_clean.csv")
bps = pd.read_csv(CLEAN_DIR / "bps_provincial_clean.csv")
sp = pd.read_csv(CLEAN_DIR / "satgas_pasti_press_clean.csv")
dir_apps = pd.read_csv(CLEAN_DIR / "ojk_direktori_playstore_clean.csv")
revs = pd.read_csv(CLEAN_DIR / "playstore_reviews_clean.csv")

# 1. Build Platform-Level Play Store Risk Profile (preserving recency sampling scope without fabricating 2023 monthly backfills)
rev_agg = revs.groupby("no_ojk", as_index=False).agg(
    sample_reviews_n=("reviewId", "count"),
    sample_mean_rating=("score", "mean"),
    sample_neg_rate_pct=("is_negative_review", lambda s: s.mean() * 100.0),
    sample_rejection_rate_pct=("kw_rejection_limit", lambda s: s.mean() * 100.0),
    sample_tech_issue_rate_pct=("kw_app_tech_issue", lambda s: s.mean() * 100.0),
    sample_cost_rate_pct=("kw_cost_interest", lambda s: s.mean() * 100.0),
    sample_collection_rate_pct=("kw_collection_terror", lambda s: s.mean() * 100.0),
    sample_dev_reply_rate_pct=("has_developer_reply", lambda s: s.mean() * 100.0),
)
df_platform_profile = dir_apps.merge(rev_agg, left_on="no", right_on="no_ojk", how="left").drop(columns=["no_ojk"])
plat_out = CLEAN_DIR / "playstore_platform_risk_profile.csv"
df_platform_profile.to_csv(plat_out, index=False)
print(f"step=platform_risk_profile | rows={len(df_platform_profile)} | with_app_meta={df_platform_profile['appId'].notna().sum()} | with_review_agg={df_platform_profile['sample_reviews_n'].notna().sum()}")

# 2. Prepare BPS 2023-2026 with causal forward-fill & explicit imputation flags (zero lookahead leakage)
bps_work = bps[bps["tahun"].between(2023, 2026)].sort_values(["prov_code", "tahun"]).copy()

ojk_sub = ojk[ojk["date"] >= "2023-01-01"].copy()
ojk_sub["tahun"] = pd.to_datetime(ojk_sub["date"]).dt.year
raw_join_check = ojk_sub[["date", "tahun", "prov_code"]].merge(bps_work, on=["tahun", "prov_code"], how="left")

bps_num_cols = [
    "ump_idr", "tpt_feb_pct", "tpt_aug_pct", "tpt_mean_pct",
    "informal_worker_pct", "upah_jam_idr", "upah_bersih_bulan_idr",
    "ipm_score", "garis_kemiskinan_idr", "kemiskinan_pct", "penduduk_miskin_ribu",
    "pengeluaran_makanan_idr", "pengeluaran_nonmakanan_idr", "pengeluaran_total_idr",
    "rasio_pengeluaran_nonmakanan", "penduduk_ribu", "laju_pertumbuhan_penduduk_pct",
    "kepadatan_penduduk_km2", "pdrb_perkapita_ribu_idr",
]
nulls_before = raw_join_check[bps_num_cols].isna().sum().to_dict()

# Fix Papua Pegunungan (prov_code 97) 2025 UMP using parent Papua (prov_code 94) 2025 UMP with explicit flag
papua_2025_ump = float(bps_work.loc[(bps_work["prov_code"] == 94) & (bps_work["tahun"] == 2025), "ump_idr"].iloc[0])
pp_2025_mask = (bps_work["prov_code"] == 97) & (bps_work["tahun"] == 2025) & (bps_work["ump_idr"].isna())
bps_work["ump_imputed_flag"] = np.where(pp_2025_mask, 1, 0)
bps_work.loc[pp_2025_mask, "ump_idr"] = papua_2025_ump
bps_work.loc[pp_2025_mask, "ump_source"] = "Parent_Papua_Carryover"

# Causal forward-fill ONLY within each province (latest published annual/Susenas figure -> 2025/2026; never backward)
ffill_cols = [
    "informal_worker_pct", "upah_jam_idr", "ipm_score",
    "garis_kemiskinan_idr", "kemiskinan_pct", "penduduk_miskin_ribu",
    "pengeluaran_makanan_idr", "pengeluaran_nonmakanan_idr", "pengeluaran_total_idr",
    "rasio_pengeluaran_nonmakanan", "laju_pertumbuhan_penduduk_pct", "pdrb_perkapita_ribu_idr",
]
bps_work["bps_ffill_annual_flag"] = 0
for col in ffill_cols:
    filled = bps_work.groupby("prov_code")[col].ffill()
    newly_filled = bps_work[col].isna() & filled.notna()
    bps_work.loc[newly_filled, "bps_ffill_annual_flag"] = 1
    bps_work[col] = filled

master = ojk_sub.merge(
    bps_work.drop(columns=["province", "region"]),
    on=["tahun", "prov_code"],
    how="left",
)
nulls_after = master[bps_num_cols].isna().sum().to_dict()
print("\nstep=bps_missingness_audit (before -> after causal fill on 1,448 panel rows)")
for c in bps_num_cols:
    if nulls_before[c] > 0 or nulls_after[c] > 0:
        print(f"  {c:32s}: before={nulls_before[c]:4d} -> after={nulls_after[c]:4d}")

# 3. Add National Monthly Context (OJK National Aggregates + Causal Trailing 6-month Satgas PASTI Enforcement)
nat_ojk = master.groupby("date", as_index=False).agg(
    nat_ojk_outstanding_miliar=("outstanding_miliar", "sum"),
    nat_ojk_rekening_aktif=("rekening_aktif", "sum"),
    nat_ojk_mean_twp90_pct=("twp90_pct", "mean"),
    nat_active_provinces=("prov_code", "count"),
)
master = master.merge(nat_ojk, on="date", how="left")
master["prov_outstanding_share_pct"] = (master["outstanding_miliar"] / master["nat_ojk_outstanding_miliar"]) * 100.0

sp_m = sp.groupby("press_month", as_index=False).agg(
    sp_press_n=("title_clean", "count"),
    sp_blocked_n=("entities_blocked_mentioned", "sum"),
)
all_months = pd.DataFrame({"date": sorted(master["date"].unique())})
all_months = all_months.merge(sp_m, left_on="date", right_on="press_month", how="left").fillna({"sp_press_n": 0, "sp_blocked_n": 0})
all_months = all_months.sort_values("date").reset_index(drop=True)
all_months["satgas_6m_press_count"] = all_months["sp_press_n"].rolling(window=6, min_periods=1).sum()
all_months["satgas_6m_blocked_entities"] = all_months["sp_blocked_n"].rolling(window=6, min_periods=1).sum()
master = master.merge(all_months[["date", "satgas_6m_press_count", "satgas_6m_blocked_entities"]], on="date", how="left")

# 4. Outlier Detection & Domain Justification (IQR, Modified Z-score MAD, IsolationForest)
audit_cols = [
    "twp90_pct", "outstanding_miliar", "rekening_aktif",
    "ump_idr", "tpt_mean_pct", "kemiskinan_pct",
    "pengeluaran_total_idr", "pdrb_perkapita_ribu_idr",
]
outlier_summary = []
for col in audit_cols:
    s = master[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    iqr_mask = (master[col] < (q1 - 1.5 * iqr)) | (master[col] > (q3 + 1.5 * iqr))
    med = s.median()
    mad = np.median(np.abs(s - med))
    mod_z = 0.6745 * (master[col] - med) / (mad if mad > 0 else 1.0)
    mad_mask = mod_z.abs() > 3.5
    if col == "twp90_pct":
        master["is_twp90_iqr_outlier"] = iqr_mask.astype(int)
    outlier_summary.append({
        "feature": col,
        "iqr_n": int(iqr_mask.sum()),
        "iqr_pct": round(float(iqr_mask.mean() * 100.0), 2),
        "mad_z_n": int(mad_mask.sum()),
        "min": round(float(s.min()), 2),
        "median": round(float(med), 2),
        "max": round(float(s.max()), 2),
    })

iso_X = master[audit_cols].fillna(master[audit_cols].median())
iso = IsolationForest(contamination=0.03, random_state=SEED)
master["is_multivar_outlier"] = (iso.fit_predict(iso_X) == -1).astype(int)

print("\nstep=outlier_audit_univariate_iqr_and_mad")
print(pd.DataFrame(outlier_summary).to_string(index=False))

print(f"\nstep=outlier_audit_multivariate_isoforest | flagged={int(master['is_multivar_outlier'].sum())} / {len(master)} ({master['is_multivar_outlier'].mean()*100:.2f}%)")
print("  top_extreme_twp90_retained:")
print(master.sort_values("twp90_pct", ascending=False).head(6)[["date", "province", "twp90_pct", "outstanding_miliar", "pdrb_perkapita_ribu_idr", "is_twp90_iqr_outlier", "is_multivar_outlier"]].to_string(index=False))
print("\n  top_extreme_outstanding_retained:")
print(master.sort_values("outstanding_miliar", ascending=False).head(5)[["date", "province", "twp90_pct", "outstanding_miliar", "rekening_aktif", "is_multivar_outlier"]].to_string(index=False))

# Export master panel clean (zero duplicate keys, ISO YYYY-MM-DD dates)
assert master.duplicated(subset=["date", "prov_code"]).sum() == 0, "Duplicate (date, prov_code) in master!"
master = master.sort_values(["date", "prov_code"]).reset_index(drop=True)
master_out = CLEAN_DIR / "master_pindar_panel_clean.csv"
master.to_csv(master_out, index=False)

print(f"\nstep=master_panel_exported | file={master_out.name} | shape={master.shape} | dates={master['date'].min()}..{master['date'].max()} ({master['date'].nunique()} mos)")
print(f"  breach_twp90_ge_5pct={int((master['twp90_pct'] >= 5.0).sum())} ({(master['twp90_pct'] >= 5.0).mean()*100:.2f}%) | elevated_ge_3pct={int((master['twp90_pct'] >= 3.0).sum())} ({(master['twp90_pct'] >= 3.0).mean()*100:.2f}%)")


step=platform_risk_profile | rows=94 | with_app_meta=69 | with_review_agg=66

step=bps_missingness_audit (before -> after causal fill on 1,448 panel rows)
  ump_idr                         : before=  12 -> after=   0
  informal_worker_pct             : before= 152 -> after=   0
  upah_jam_idr                    : before= 152 -> after=   0
  upah_bersih_bulan_idr           : before= 408 -> after= 408
  ipm_score                       : before= 608 -> after=   0
  garis_kemiskinan_idr            : before= 152 -> after=   0
  kemiskinan_pct                  : before= 152 -> after=   0
  penduduk_miskin_ribu            : before= 152 -> after=   0
  pengeluaran_makanan_idr         : before= 152 -> after=   0
  pengeluaran_nonmakanan_idr      : before= 152 -> after=   0
  pengeluaran_total_idr           : before= 152 -> after=   0
  rasio_pengeluaran_nonmakanan    : before= 152 -> after=   0
  pdrb_perkapita_ribu_idr         : before= 152 -> after=   0



step=outlier_audit_univariate_iqr_and_mad
                feature  iqr_n  iqr_pct  mad_z_n        min     median        max
              twp90_pct     54     3.73       45       0.03       1.72      11.58
     outstanding_miliar    210    14.50      210       1.05     648.16   24829.91
         rekening_aktif    209    14.43      203     446.00  140519.50 6427757.00
                ump_idr     40     2.76       16 1958170.00 3164771.50 5729876.00
           tpt_mean_pct      0     0.00        0       1.25       4.24       7.74
         kemiskinan_pct     56     3.87       44       3.57       9.40      31.32
  pengeluaran_total_idr     56     3.87       56  961372.00 1459088.00 2962412.00
pdrb_perkapita_ribu_idr    216    14.92      176   18105.00   69352.00  367687.00

step=outlier_audit_multivariate_isoforest | flagged=44 / 1448 (3.04%)
  top_extreme_twp90_retained:
      date            province  twp90_pct  outstanding_miliar  pdrb_perkapita_ribu_idr  is_twp90_iqr_outlier  is_multi


step=master_panel_exported | file=master_pindar_panel_clean.csv | shape=(1448, 69) | dates=2023-01-01..2026-04-01 (40 mos)
  breach_twp90_ge_5pct=32 (2.21%) | elevated_ge_3pct=190 (13.12%)
